# Tier B light labs


This notebook is Part II‑B from
`smooth_manifolds_lie_groups_and_the_exponential_map.ipynb`:
Minkowski spacetime as a Lorentzian manifold, and a $U(1)$ Berry
phase on a latitude of the Bloch sphere. The wording below is
that text. Examples (M1)–(M3), charts and atlases, Lie groups
and the exponential map, and Tiers A and C stay in the mega
notebook.

The code cells check the light cone in $\mathbb{R}^{1,1}$ and
the latitude Berry phase. There is no sibling demo.


In [ ]:
# Setup — foundations notebook (numpy / scipy / matplotlib only)
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm, det, eigvalsh
from scipy.linalg import expm, logm

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (5.5, 4.0), "font.size": 11})

print("numpy", np.__version__)


# Part II‑B — Light labs


Same three‑part shape as Tier A (background → geometry → applications),
with shorter code.


## B1. Minkowski spacetime as a smooth manifold


### Background


Special relativity is not “Euclidean geometry with an extra
coordinate.” Events form a four‑dimensional set of points, and the
physically meaningful notion of length is the **Minkowski** interval,
which treats time differently from space. Light rays sit on the
boundary between travel that is possible for massive particles and
travel that is not.


### Geometry


$\mathbb{R}^{1,3}$ with metric $\eta=\mathrm{diag}(-1,1,1,1)$ (or the
opposite sign convention) is a smooth manifold diffeomorphic to
ordinary $\mathbb{R}^4$, but the **metric** distinguishes timelike,
null, and spacelike vectors:

$$
v\cdot v = \eta_{\mu\nu}v^\mu v^\nu
  \quad
  \begin{cases}
  <0 & \text{timelike (massive worldlines)},\\
  =0 & \text{null (light rays)},\\
  >0 & \text{spacelike}.
  \end{cases}
$$

Lorentz transformations are the Lie group $\mathrm{O}(1,3)$ preserving
$\eta$; adjoining translations gives the Poincaré group.


### Where it shows up


- **Particle kinematics:** energy–momentum relations, decay thresholds,
  collider frame changes.
- **Field theory:** causal structure of wave equations; light cones
  bound signal propagation.
- **General relativity:** curved spacetime is a Lorentzian manifold;
  Minkowski space is the flat local model (Wald, Carroll).

> **Further reading.** S. Carroll, *Spacetime and Geometry*, Addison
> Wesley / Cambridge reprint, Ch. 1–2. R. Wald, *General Relativity*,
> Chicago, 1984, Ch. 1–2 (for the manifold + metric viewpoint).


In [ ]:
# Light cone in 1+1 Minkowski space

eta = np.diag([-1.0, 1.0])
xs = np.linspace(-2, 2, 400)
fig, ax = plt.subplots()
ax.fill_between(xs, xs, -xs, where=(xs >= 0), alpha=0.08, color="C0")
ax.fill_between(xs, xs, -xs, where=(xs <= 0), alpha=0.08, color="C0")
ax.plot(xs, xs, "k-", lw=1, label=r"null $t=\pm x$")
ax.plot(xs, -xs, "k-", lw=1)
ax.scatter([0], [0], c="C3", zorder=5)
ax.set_aspect("equal")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$t$")
ax.set_title(r"Minkowski plane $\mathbb{R}^{1,1}$: light cone")
ax.legend(fontsize=9)
plt.show()

# Classify a few vectors
def kind(v):
    s = v @ eta @ v
    if abs(s) < 1e-12:
        return "null"
    return "timelike" if s < 0 else "spacelike"

for v in [(1, 0), (0, 1), (1, 1), (1, 2)]:
    print(v, kind(np.array(v, dtype=float)))


**Takeaway box — B1.**  
Manifold: spacetime (here flat Minkowski). Extra structure: Lorentzian
metric. Isometry Lie group: Lorentz / Poincaré.


## B2. $U(1)$ bundles and Berry phase (sketch with code)


### Background


If you slowly cycle the knobs of a Hamiltonian and return them to the
start, the quantum state need not return with only the usual dynamical
phase $e^{-iEt/\hbar}$. An extra **geometric** phase can remain — it
depends on the *path* in parameter space, not on how fast you traverse
it (in the adiabatic idealisation). Berry’s insight was that this phase
is the holonomy of a natural connection on the bundle of eigenstates.


### Geometry


A **principal $U(1)$ bundle** over a base $M$ assigns a circle's worth
of phases to each point of $M$, glued consistently. In quantum
mechanics, a normalised eigenstate $|n(\mathbf{R})\rangle$ depending on
slow parameters $\mathbf{R}$ defines a line bundle over the parameter
manifold; transporting the state around a closed loop accumulates a
geometric **Berry phase**.

For a spin‑1/2 in a magnetic field $\mathbf{B}=B\,\hat{\mathbf{n}}$, a
loop of $\hat{\mathbf{n}}$ on the Bloch sphere enclosing solid angle
$\Omega$ gives Berry phase

$$
\gamma = -\frac{\Omega}{2}
$$

for the ground state (monopole curvature of strength $1/2$ at the
origin of $\mathbf{B}$‑space).


### Where it shows up


- **Molecular / condensed‑matter physics:** electronic states that pick
  up Berry phases when nuclear coordinates cycle; anomalous velocity
  terms in solids.
- **Quantum control:** holonomic gates that encode computation in
  geometric phases, aiming for robustness to timing noise.
- **Topology:** Chern numbers and related invariants are integrals of
  Berry curvature — the same geometry at a global scale.

> **Further reading.** M. Nakahara, *Geometry, Topology and Physics*,
> 2nd ed., Taylor & Francis, 2003, Ch. 9–10 (bundles, connections,
> Berry phase §10.6). M. V. Berry, “Quantal phase factors accompanying
> adiabatic changes,” *Proc. R. Soc. Lond. A* **392** (1984) 45–57.


In [ ]:
# Berry phase for spin-1/2: loop at fixed latitude theta on the Bloch sphere
# Analytic: gamma = -pi (1 - cos(theta))  [= -Omega/2, Omega = 2pi(1-cos theta)]

def ground_state(theta, phi):
    # H ~ n · sigma; ground state for spin along -n if we take energy -|B|
    # Standard: |n-; lower> = (sin(theta/2) e^{-i phi}, -cos(theta/2)) up to phase
    return np.array(
        [np.sin(theta / 2) * np.exp(-1j * phi), -np.cos(theta / 2)],
        dtype=complex,
    )

def berry_phase_latitude(theta, n_steps=500):
    phis = np.linspace(0, 2 * np.pi, n_steps, endpoint=False)
    states = np.stack([ground_state(theta, phi) for phi in phis], axis=0)
    # Remove discontinuous chart jumps by fixing local phase continuity
    for k in range(1, n_steps):
        overlap = np.vdot(states[k - 1], states[k])
        if abs(overlap) > 0:
            states[k] *= np.exp(-1j * np.angle(overlap))
    # Close the loop
    total = 1.0 + 0j
    for k in range(n_steps):
        total *= np.vdot(states[k], states[(k + 1) % n_steps])
    # After phase alignment, open-path product ≈ 1; geometric phase from closure:
    closure = np.vdot(states[-1], states[0])
    # Standard discrete Berry: arg of product of overlaps around the loop
    prod = 1.0 + 0j
    for k in range(n_steps):
        prod *= np.vdot(states[k], states[(k + 1) % n_steps])
    return np.angle(prod)

theta = np.pi / 3
gamma_num = berry_phase_latitude(theta)
gamma_an = -np.pi * (1 - np.cos(theta))
print("Berry phase numeric:", gamma_num)
print("Berry phase analytic -π(1-cos θ):", gamma_an)
print("abs error:", abs(((gamma_num - gamma_an + np.pi) % (2 * np.pi)) - np.pi))


**Takeaway box — B2.**  
Base manifold: parameter space (here $S^2$ of field directions).
Bundle: $U(1)$ phases of eigenstates. Connection: Berry connection;
curvature → solid angle / monopole.
